In [33]:
import requests
import pandas as pd

In [6]:
def fetch_journals_by_topic(topic_id: str, max_results: int = None) -> list:
    params = {
        "filter": f"type:journal,topics.id:{topic_id}",
        "sort": "works_count:desc",
        "select": "display_name,works_count,issn_l",
        "per_page": 200,
        "cursor": "*",
    }

    results = []
    while True:
        r = requests.get("https://api.openalex.org/sources", params=params)
        data = r.json()
        results.extend(data["results"])
        if max_results and len(results) >= max_results:
            results = results[:max_results]
            break
        next_cursor = data["meta"].get("next_cursor")
        if not next_cursor:
            break
        params["cursor"] = next_cursor

    return results

### 1. List of topics (chemistry)

In [7]:
r = requests.get("https://api.openalex.org/topics", params={"search": "chemistry", "per_page": 200})

In [12]:
data = r.json()
topic_dict = {}
for t in data["results"]:
    tid = t["id"].split("/")[-1]
    topic_dict[tid] =  t["display_name"]
print(f"Total topics: {len(topic_dict)}")

Total topics: 128


### 2. List of journals

In [28]:
journal_dict = {}
for n, tid in enumerate(topic_dict.keys()):
    
    res = fetch_journals_by_topic(tid)
    print(f"{n+1} { topic_dict[tid]} -> Journals: {len(res)}")
    
    for j in res:
        journal_dict[j['display_name']] = j['works_count']
        
print(f"Total journals: {len(journal_dict)}")
print(f"Total papers: {sum(journal_dict.values())}")

1 Atmospheric chemistry and aerosols -> Journals: 1068
2 Analytical Chemistry and Chromatography -> Journals: 2478
3 DNA and Nucleic Acid Chemistry -> Journals: 706
4 Analytical chemistry methods development -> Journals: 1187
5 Various Chemistry Research Topics -> Journals: 726
6 Radioactive element chemistry and processing -> Journals: 915
7 Porphyrin and Phthalocyanine Chemistry -> Journals: 331
8 History and advancements in chemistry -> Journals: 940
9 Fluorine in Organic Chemistry -> Journals: 225
10 Inorganic and Organometallic Chemistry -> Journals: 1311
11 Analytical Chemistry and Sensors -> Journals: 1378
12 Click Chemistry and Applications -> Journals: 346
13 Synthetic Organic Chemistry Methods -> Journals: 266
14 Fullerene Chemistry and Applications -> Journals: 138
15 Carbohydrate Chemistry and Synthesis -> Journals: 430
16 Supramolecular Chemistry and Complexes -> Journals: 122
17 Chemistry and Chemical Engineering -> Journals: 711
18 Lignin and Wood Chemistry -> Journals: 

In [37]:
tmp = pd.DataFrame()
tmp["journal"] = journal_dict.keys()
tmp["count"] = journal_dict.values()
tmp.to_csv("backlog/jpurnal_list.csv", index=False)